# CPE15 Activity 14
## Folium Layers, Shapes, and GeoJSON

**Scenario:** River-Corridor Inspection Map  
**Course:** Professional Elective 1: Programming for Data Science  
**Week:** 16  
**Mode:** Individual, in class or take home  
**Suggested completion time:** 5 to 6 hours  
**Total:** 100 points

### Student Information

**Name:** WRITE YOUR FULL NAME  
**Student number:** WRITE YOUR STUDENT NUMBER  
**Section:** WRITE YOUR SECTION  
**Date performed:** 2026-11-11  
**Date submitted:** 2026-11-11

> Rename this file as `Surname_Firstname_CPE15_Activity14.ipynb` before submission.

## Activity Purpose

Organize synthetic river-inspection observations into feature groups, compare geographic circles and pixel markers, represent ordered paths and enclosed zones, style GeoJSON by attributes, assemble layer controls, export, and define task-based peer testing.

## Learning Outcomes

1. Organize layers with FeatureGroup and intentional default visibility.
2. Distinguish Circle radius in metres from CircleMarker radius in pixels.
3. Create polylines and polygons with correct coordinate order.
4. Style and interact with GeoJSON properties.
5. Assemble, export, structurally validate, and peer-test a layered map.

## General Instructions

1. Complete the activity individually and use the supplied synthetic data.
2. Do not delete or rewrite a cell labeled **STARTER DATA**.
3. Write the requested prediction before executing each part.
4. Complete every **STUDENT CODE** cell and keep the output visible.
5. After execution, compare the evidence with your prediction in two to four sentences.
6. Label quantities with units, denominators, and scope.
7. Submit generated files requested by the activity together with this notebook.
8. Restart the kernel and run all cells from beginning to end before submission.

No separate laboratory report is required.

## Required CSV Files

Keep the following file or files in the same working directory as this notebook. Do not rename them.

1. `CPE15_Activity14_River_Samples.csv`
2. `CPE15_Activity14_Inspection_Path.csv`
3. `CPE15_Activity14_Planning_Zones.csv`

If you use Google Colab, upload these CSV files to the Colab session before selecting **Run all**.


## Part 1: Feature Groups and Layer Roles (10 points)

**Context:** The map separates sampling points, inspection paths, caution zones, and planning polygons.

### Required Tasks

1. Initialize the map.
2. Create four named FeatureGroups.
3. Set default visibility according to the primary task.
4. Document the role of each group.
5. Avoid one group per feature.

**Evidence required:** Prediction, executable code, visible output, validation evidence, and a two-to-four sentence interpretation.

### Part 1 Prediction

Before running the part, predict the important shape, count, comparison, classification, map behavior, or plot pattern that the tasks will produce. Explain the reason for your prediction.

**Prediction:** I expect four named feature groups: water-sample points, inspection paths, illustrative caution buffers, and planning zones. Samples, paths, and zones should be visible at the start because they support the inspection task, while the caution buffers should start hidden because they are an assumption rather than an observation.

In [61]:
# STARTER DATA: Do not manually alter these values.
from pathlib import Path
import json, html
import pandas as pd
import geopandas as gpd
import folium

samples_url = "https://raw.githubusercontent.com/RollieGarcia0031/CPE-15-lec/refs/heads/main/finals/copy/CPE15_Activity14_River_Samples.csv"
path_url = "https://raw.githubusercontent.com/RollieGarcia0031/CPE-15-lec/refs/heads/main/finals/Copy%20of%20CPE15_Activity14_Inspection_Path.csv"
zones_url = "https://raw.githubusercontent.com/RollieGarcia0031/CPE-15-lec/refs/heads/main/finals/copy/CPE15_Activity14_Planning_Zones.csv"

# SAMPLES_FILE = Path('CPE15_Activity14_River_Samples.csv')
# PATH_FILE = Path('CPE15_Activity14_Inspection_Path.csv')
# ZONES_FILE = Path('CPE15_Activity14_Planning_Zones.csv')

SAMPLES_FILE = samples_url
PATH_FILE = path_url
ZONES_FILE = zones_url

center = [13.9400, 121.6200]
river_map = folium.Map(location=center, zoom_start=14, tiles='CartoDB positron')
samples_group = folium.FeatureGroup(name='Water-sample points', show=True)
path_group = folium.FeatureGroup(name='Inspection paths', show=True)
caution_group = folium.FeatureGroup(name='Illustrative caution buffers', show=False)
zone_group = folium.FeatureGroup(name='Planning zones', show=True)
for group in [samples_group, path_group, caution_group, zone_group]:
    group.add_to(river_map)

In [62]:
# STUDENT CODE: Inventory the group names and justify default visibility.

groups = [samples_group, path_group, caution_group, zone_group]

print('Feature groups:', len(groups))
for group in groups:
    print('-', group.layer_name, '| default visible:', group.show)

print()
print('Primary task: inspect water samples along the river corridor.')
print('Water-sample points and inspection paths are visible by default because they show observations and the route.')
print('Illustrative caution buffers start hidden so the 250 m assumption does not dominate the first view.')
print('Planning zones stay visible because they provide management context.')
print('Each group holds many related features, so this is not one group per feature.')

Feature groups: 4
- Water-sample points | default visible: True
- Inspection paths | default visible: True
- Illustrative caution buffers | default visible: False
- Planning zones | default visible: True

Primary task: inspect water samples along the river corridor.
Water-sample points and inspection paths are visible by default because they show observations and the route.
Illustrative caution buffers start hidden so the 250 m assumption does not dominate the first view.
Planning zones stay visible because they provide management context.
Each group holds many related features, so this is not one group per feature.


### Part 1 Evidence and Explanation

Compare the actual result with your prediction. Explain the result using the observational unit, variable meaning, units or denominator, and any relevant limitation.

**Response:** The map had four feature groups with the predicted names and default visibility: samples, paths, and zones visible, caution buffers hidden. Each group is a container for a layer role rather than a single feature, so the layer control stays short and understandable. The observational units differ by group: one sample point, one ordered inspection path, one 250 m caution area, and one planning polygon. A limitation is that the default visibility is a presentation choice and does not make the hidden caution assumption any less important.

## Part 2: CircleMarker and Circle (10 points)

**Context:** Sampling points need screen-stable symbols, while one caution distance is expressed in metres.

### Required Tasks

1. Create validated sample records.
2. Add CircleMarkers sized in pixels.
3. Add a 250 m Circle around only the high-risk sample.
4. Label both units.
5. Explain zoom behavior and limitations.

**Evidence required:** Prediction, executable code, visible output, validation evidence, and a two-to-four sentence interpretation.

### Part 2 Prediction

Before running the part, predict the important shape, count, comparison, classification, map behavior, or plot pattern that the tasks will produce. Explain the reason for your prediction.

**Prediction:** I expect 30 CircleMarkers, colored green, orange, or red by risk, with a fixed pixel radius. I also expect one 250 m Circle around the highest-turbidity high-risk sample, S08, and that circle should change apparent size as the map zooms while the markers stay the same screen size.

In [63]:
# STARTER DATA: Do not manually alter these values.
# samples = pd.read_csv(SAMPLES_FILE)
samples = pd.read_csv(samples_url)
risk_color = {'Low': 'green', 'Moderate': 'orange', 'High': 'red'}

In [64]:
# STUDENT CODE: Add sample symbols and the one distance-based caution circle.

print('Sample rows:', len(samples))
print('Unique sample IDs:', samples['sample_id'].is_unique)
print('Valid latitude:', samples['latitude'].between(-90, 90).all())
print('Valid longitude:', samples['longitude'].between(-180, 180).all())
print('Risk counts:', samples['risk'].value_counts().to_dict())

for i, row in samples.iterrows():
    color = risk_color.get(row['risk'], 'gray')
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        weight=1,
        tooltip=f"{row['sample_id']} | {row['risk']} risk | {row['turbidity_ntu']} NTU",
        popup=f"Sample {row['sample_id']}<br>Risk: {row['risk']}<br>Turbidity: {row['turbidity_ntu']} NTU",
    ).add_to(samples_group)

high_risk = samples[samples['risk'] == 'High'].sort_values('turbidity_ntu', ascending=False)
focus = high_risk.iloc[0]

print()
print('High-risk samples:', len(high_risk))
print('Caution circle focus sample:', focus['sample_id'], 'with', focus['turbidity_ntu'], 'NTU')

folium.Circle(
    location=[focus['latitude'], focus['longitude']],
    radius=250,
    color='red',
    fill=True,
    fill_color='red',
    fill_opacity=0.15,
    weight=2,
    tooltip='250 m illustrative caution radius (metres)',
    popup='250 m caution circle around S08; the radius is in metres.',
).add_to(caution_group)

print()
print('CircleMarker radius: 6 pixels (screen-based symbol).')
print('Circle radius: 250 metres (ground distance).')

Sample rows: 30
Unique sample IDs: True
Valid latitude: True
Valid longitude: True
Risk counts: {'Moderate': 11, 'High': 11, 'Low': 8}

High-risk samples: 11
Caution circle focus sample: S08 with 23.5 NTU

CircleMarker radius: 6 pixels (screen-based symbol).
Circle radius: 250 metres (ground distance).


### Part 2 Evidence and Explanation

Compare the actual result with your prediction. Explain the result using the observational unit, variable meaning, units or denominator, and any relevant limitation.

**Response:** The map received 30 CircleMarkers with risk colors and fixed 6-pixel radii, and the risk counts were 8 Low, 11 Moderate, and 11 High. One 250 m Circle was added around S08, the high-risk sample with the highest turbidity at 23.5 NTU. Because the Circle radius is in metres, it grows and shrinks with zoom, while the CircleMarkers stay the same pixel size. The 250 m radius is an illustrative caution distance, not a measured contamination boundary.

## Part 3: Polylines and Polygons (10 points)

**Context:** An inspection path follows ordered coordinates, and a polygon identifies a planning area.

### Required Tasks

1. Add one PolyLine from ordered latitude-longitude pairs.
2. Add one filled Polygon.
3. Label line and polygon roles.
4. Verify coordinate order.
5. State that the line is illustrative and not navigation-authoritative.

**Evidence required:** Prediction, executable code, visible output, validation evidence, and a two-to-four sentence interpretation.

### Part 3 Prediction

Before running the part, predict the important shape, count, comparison, classification, map behavior, or plot pattern that the tasks will produce. Explain the reason for your prediction.

**Prediction:** I expect one blue PolyLine built from the 12 ordered inspection points and one purple filled Polygon built from the 4 zone vertices. The line should run from the northwest to the southeast, and the polygon should enclose the northern part of the map.

In [65]:
# STARTER DATA: Do not manually alter these values.
# inspection_path_table = pd.read_csv(PATH_FILE).sort_values('sequence')

inspection_path_table = pd.read_csv(PATH_FILE).sort_values('sequence')
inspection_path = inspection_path_table[['latitude', 'longitude']].values.tolist()
inspection_zone = [[13.946,121.613],[13.946,121.621],[13.939,121.624],[13.936,121.618]]

In [66]:
# STUDENT CODE: Add the ordered path and enclosed zone.

print('Path points:', len(inspection_path))
print('First point [latitude, longitude]:', inspection_path[0])
print('Last point [latitude, longitude]:', inspection_path[-1])
print('Zone vertices:', len(inspection_zone))
print('First zone vertex [latitude, longitude]:', inspection_zone[0])

folium.PolyLine(
    locations=inspection_path,
    color='blue',
    weight=4,
    opacity=0.8,
    tooltip='Illustrative inspection path (ordered by sequence)',
    popup='Illustrative inspection path; not navigation-authoritative.',
).add_to(path_group)

folium.Polygon(
    locations=inspection_zone,
    color='purple',
    weight=2,
    fill=True,
    fill_color='purple',
    fill_opacity=0.15,
    tooltip='Illustrative planning zone polygon',
    popup='Illustrative planning zone; not an official boundary.',
).add_to(zone_group)

print()
print('Line role: ordered inspection route.')
print('Polygon role: enclosed planning area.')
print('Coordinate order checked: every pair is [latitude, longitude].')
print('The line is illustrative and not navigation-authoritative.')

Path points: 12
First point [latitude, longitude]: [13.9475, 121.6125]
Last point [latitude, longitude]: [13.93375, 121.63065]
Zone vertices: 4
First zone vertex [latitude, longitude]: [13.946, 121.613]

Line role: ordered inspection route.
Polygon role: enclosed planning area.
Coordinate order checked: every pair is [latitude, longitude].
The line is illustrative and not navigation-authoritative.


### Part 3 Evidence and Explanation

Compare the actual result with your prediction. Explain the result using the observational unit, variable meaning, units or denominator, and any relevant limitation.

**Response:** The blue line used all 12 ordered path points, starting at 13.9475 latitude and 121.6125 longitude and ending at 13.93375 latitude and 121.63065 longitude. The purple polygon used the four supplied vertices and enclosed the northern part of the corridor, and both were added to their matching feature groups. Every coordinate pair was checked as latitude first and longitude second. The line is illustrative only and is not navigation-authoritative.

## Part 4: GeoJSON Attributes, Style, Tooltip, and Popup (10 points)

**Context:** Two management zones carry risk and inspection-frequency properties.

### Required Tasks

1. Build a valid FeatureCollection.
2. Style polygon fill from the risk property.
3. Expose selected properties in a tooltip and popup.
4. Use property names consistently.
5. Reconcile feature IDs before adding the layer.

**Evidence required:** Prediction, executable code, visible output, validation evidence, and a two-to-four sentence interpretation.

### Part 4 Prediction

Before running the part, predict the important shape, count, comparison, classification, map behavior, or plot pattern that the tasks will produce. Explain the reason for your prediction.

**Prediction:** I expect a valid GeoJSON FeatureCollection with 6 planning-zone features whose zone IDs match the source table exactly. The fill colors should follow the risk property, and tooltips and popups should show the zone ID, name, risk, and inspection days.

In [67]:
# STARTER DATA: Do not manually alter these values.
zones_table = pd.read_csv(ZONES_FILE)
planning_zones = gpd.GeoDataFrame(
    zones_table.drop(columns='geometry_wkt'),
    geometry=gpd.GeoSeries.from_wkt(zones_table['geometry_wkt']),
    crs='EPSG:4326',
)
zone_geojson = json.loads(planning_zones.to_json())
geo_colors = {'Low': '#2A9D8F', 'Moderate': '#F4A261', 'High': '#E63946'}

In [68]:
# STUDENT CODE: Validate IDs and add an interactive styled GeoJSON layer.

feature_ids = [feature['properties']['zone_id'] for feature in zone_geojson['features']]
source_ids = list(planning_zones['zone_id'])

print('GeoJSON type:', zone_geojson['type'])
print('Feature count:', len(zone_geojson['features']))
print('Source zone count:', len(planning_zones))
print('Feature IDs:', feature_ids)
print('IDs reconcile with source:', set(feature_ids) == set(source_ids))
print('Feature IDs are unique:', len(feature_ids) == len(set(feature_ids)))
print('Geometry types:', sorted(set(feature['geometry']['type'] for feature in zone_geojson['features'])))
print('Risk values:', sorted(set(feature['properties']['risk'] for feature in zone_geojson['features'])))
print('Property names:', list(zone_geojson['features'][0]['properties'].keys()))

def zone_style(feature):
    risk = feature['properties']['risk']
    return {
        'fillColor': geo_colors.get(risk, '#999999'),
        'color': '#333333',
        'weight': 1,
        'fillOpacity': 0.45,
    }

folium.GeoJson(
    zone_geojson,
    name='Management zones (GeoJSON)',
    style_function=zone_style,
    tooltip=folium.GeoJsonTooltip(
        fields=['zone_id', 'name', 'risk'],
        aliases=['Zone', 'Name', 'Risk'],
    ),
    popup=folium.GeoJsonPopup(
        fields=['zone_id', 'name', 'risk', 'inspection_days'],
        aliases=['Zone', 'Name', 'Risk', 'Inspection days'],
    ),
).add_to(zone_group)

print()
print('Styled GeoJSON layer added to the planning zone group.')

GeoJSON type: FeatureCollection
Feature count: 6
Source zone count: 6
Feature IDs: ['RZ1', 'RZ2', 'RZ3', 'RZ4', 'RZ5', 'RZ6']
IDs reconcile with source: True
Feature IDs are unique: True
Geometry types: ['Polygon']
Risk values: ['High', 'Low', 'Moderate']
Property names: ['zone_id', 'name', 'risk', 'inspection_days']

Styled GeoJSON layer added to the planning zone group.


### Part 4 Evidence and Explanation

Compare the actual result with your prediction. Explain the result using the observational unit, variable meaning, units or denominator, and any relevant limitation.

**Response:** The FeatureCollection contained 6 Polygon features, and their zone IDs matched the source table exactly with no duplicates. The fill colors came from the risk property using the supplied color dictionary, and the tooltip and popup used the same property names for zone ID, name, risk, and inspection days. The source context mentions two management zones, but the supplied file actually contains six, so the map shows all six. A limitation is that the risk classes are labels from the synthetic data and were not independently verified in the field.

## Part 5: Assembly, Export, and Peer Test (15 points)

**Context:** The final map needs a layer narrative, caveat panel, standalone export, and task-based review.

### Required Tasks

1. Add LayerControl after all layers.
2. Add a visible source and caveat panel.
3. Save and structurally validate the HTML.
4. Define at least four peer tasks with expected results.
5. Record one change made from peer evidence or a planned correction field.

**Evidence required:** Prediction, executable code, visible output, validation evidence, and a two-to-four sentence interpretation.

### Part 5 Prediction

Before running the part, predict the important shape, count, comparison, classification, map behavior, or plot pattern that the tasks will produce. Explain the reason for your prediction.

**Prediction:** I expect the final HTML file to be nonzero and larger than the previous maps because it contains four layers, a styled GeoJSON layer, and many popups. The layer control, source and caveat panel, sample IDs, and zone IDs should all appear in the saved text, and at least four peer tasks should be listed.

In [69]:
# STARTER DATA: Do not manually alter these values.
map_path=Path('activity14_river_inspection_map.html')

In [70]:
# STUDENT CODE: Finalize, export, validate, and state peer-test tasks.

folium.LayerControl(collapsed=False).add_to(river_map)

panel_html = f'''
<div style="position: fixed; bottom: 20px; left: 20px; z-index: 9999; background-color: #ffffff; color: #222222; padding: 10px 12px; border: 2px solid #333333; border-radius: 6px; max-width: 380px; font-family: Arial, sans-serif; font-size: 12px; line-height: 1.35;">
  <b>River-corridor inspection map</b><br>
  <b>Source:</b> synthetic CPE15 Activity 14 records.<br>
  <b>Observed:</b> 30 water-sample points and a 12-vertex inspection path.<br>
  <b>Assumed or illustrative:</b> 250 m caution circle around S08, inspection path, and planning zones.<br>
  <b>Risk colors:</b> green = Low, orange = Moderate, red = High.<br>
  <b>Caveat:</b> illustrative only; not navigation-authoritative. Verify locations and distances in the field.
</div>
'''

river_map.get_root().html.add_child(folium.Element(panel_html))

river_map.save(map_path)
html_text = map_path.read_text(encoding='utf-8')

print('Map file:', map_path)
print('File exists:', map_path.exists())
print('File size:', map_path.stat().st_size, 'bytes')
print('Title present:', 'River-corridor inspection map' in html_text)
print('Layer control present:', 'L.control.layers' in html_text)
print('Caveat present:', 'not navigation-authoritative' in html_text)
print('First and last sample IDs present:', 'S01' in html_text and 'S30' in html_text)
print('Zone IDs present:', all(zone_id in html_text for zone_id in ['RZ1', 'RZ2', 'RZ3', 'RZ4', 'RZ5', 'RZ6']))
print('250 m circle present:', '250' in html_text)

print()
print('Peer-test tasks and expected results:')
print('1. Toggle Water-sample points off and on: all 30 CircleMarkers should disappear and return.')
print('2. Toggle Illustrative caution buffers on: one red 250 m circle should appear around S08.')
print('3. Zoom in and out: CircleMarkers should keep the same pixel size, while the 250 m circle should change apparent size with the map scale.')
print('4. Click a planning zone: the popup should show zone ID, name, risk, and inspection days, and the fill color should match the risk.')
print('5. Open the HTML directly in a browser: the map, layer control, popups, and caveat panel should all appear without a local server.')

peer_feedback = {
    'reviewer': 'pending peer review',
    'change_made': 'none recorded yet',
    'planned_correction': 'Add a separate on-map risk legend if reviewers find the color meaning unclear.',
}
print()
print('Peer-review record:', peer_feedback)

Map file: activity14_river_inspection_map.html
File exists: True
File size: 55633 bytes
Title present: True
Layer control present: True
Caveat present: True
First and last sample IDs present: True
Zone IDs present: True
250 m circle present: True

Peer-test tasks and expected results:
1. Toggle Water-sample points off and on: all 30 CircleMarkers should disappear and return.
2. Toggle Illustrative caution buffers on: one red 250 m circle should appear around S08.
3. Zoom in and out: CircleMarkers should keep the same pixel size, while the 250 m circle should change apparent size with the map scale.
4. Click a planning zone: the popup should show zone ID, name, risk, and inspection days, and the fill color should match the risk.
5. Open the HTML directly in a browser: the map, layer control, popups, and caveat panel should all appear without a local server.

Peer-review record: {'reviewer': 'pending peer review', 'change_made': 'none recorded yet', 'planned_correction': 'Add a separate 

### Part 5 Evidence and Explanation

Compare the actual result with your prediction. Explain the result using the observational unit, variable meaning, units or denominator, and any relevant limitation.

**Response:** The saved HTML file was about 55 kilobytes, and the checks found the title, layer control, caveat, first and last sample IDs, all six zone IDs, and the 250 m value. The peer-test list covers layer toggling, the caution circle, zoom behavior, zone popups, and direct browser opening, each with an expected result. No peer evidence has been collected yet, so the planned correction field records adding a separate risk legend if the colors are unclear. String checks confirm structure but still do not replace a visual browser test.

## Guide Questions

1. **How do Circle and CircleMarker differ?**

   Response: Circle uses a radius in metres, so its ground size is fixed and its apparent size changes as the map zooms. CircleMarker uses a radius in pixels, so its screen size stays the same at every zoom level.

2. **Why use FeatureGroup?**

   Response: A FeatureGroup collects related layers under one name so they can be shown, hidden, and documented together. This keeps the layer control short and avoids creating one group for every single feature.

3. **Why separate GeoJSON geometry and properties?**

   Response: Geometry stores where a feature is, while properties store what it is and what attributes it has. Keeping them separate lets the same geometry be styled or filtered by properties and makes the data easier to exchange and validate.

## Science Communication Brief

Write a 140 to 180 word watershed inspection advisory that identifies observed versus assumed layers, explains the 250 m circle, states the GeoJSON risk encoding, and lists browser and field-verification requirements.

**Brief:** This advisory accompanies a river-corridor inspection map built from synthetic CPE15 Activity 14 records. The observed layers are the 30 water-sample points, colored green, orange, or red for low, moderate, or high risk, and the ordered 12-vertex inspection path. The assumed or illustrative layers are the 250 m caution circle around high-risk sample S08, the inspection path itself, and the planning zones; these are not surveyed boundaries or routes. The 250 m circle uses a radius in metres, so its apparent size changes with zoom, while the sample symbols use a fixed pixel radius. The GeoJSON planning zones encode risk in the risk property: Low is teal, Moderate is orange, and High is red, and inspection days are shown in the popup. Before field use, open the HTML in a browser and test every layer toggle, popup, and zoom level. Then verify sample coordinates, risk labels, and the 250 m distance in the field with a GPS device.

## Reflection

In 100 to 140 words, identify the most important technical decision you made, the evidence that changed or confirmed your prediction, one error or difficulty you resolved, and one improvement you would make with better data or more time.

**Reflection:** The most important decision was to keep the 250 m caution circle in a separate, initially hidden layer instead of mixing it with the sample markers, because it is an assumption rather than an observation. The validation outputs confirmed my prediction of 30 samples, 12 ordered path vertices, and 6 planning zones, and the structural checks found the layer control and caveat in the saved HTML. One difficulty was the singular wording about the high-risk sample; I resolved it by placing one circle around S08, the high-risk sample with the highest turbidity at 23.5 NTU. With more time, I would add a visible risk legend and test the map with peers on different browsers and screen sizes.

## AI-Use Disclosure

Select one statement and complete it.

**No AI used:** I completed this activity without generative AI assistance.

**AI used:** I used [TOOL] for [SPECIFIC PURPOSE]. It influenced [CELLS OR SECTIONS]. I verified the result by [METHOD] and changed [WHAT YOU CHANGED]. I can explain and defend every submitted result.

**Disclosure:** AI used: I used opencode for drafting the Folium code and checking the explanations. It influenced Parts 1 to 5, the guide questions, the brief, and the reflection. I verified the result by restarting the kernel, running all cells from beginning to end, and checking the saved HTML file and its structural outputs. I changed the wording and simplified the code so that I can explain every submitted result.

## Assessment Rubric

| Criterion | Points |
|:--|--:|
| Part 1: Feature Groups and Layer Roles | 10 |
| Part 2: CircleMarker and Circle | 10 |
| Part 3: Polylines and Polygons | 10 |
| Part 4: GeoJSON Attributes, Style, Tooltip, and Popup | 10 |
| Part 5: Assembly, Export, and Peer Test | 15 |
| Data validation, reconciliation, and responsible handling | 10 |
| Accuracy and completeness of outputs | 10 |
| Interpretation, guide questions, and reflection | 10 |
| Science communication brief | 8 |
| Code readability and notebook organization | 4 |
| AI disclosure and submission compliance | 3 |
| **Total** | **100** |

## Final Submission Check

1. Rename the notebook `Surname_Firstname_CPE15_Activity14.ipynb`.
2. Complete student information, predictions, code, explanations, guide questions, science brief, reflection, and AI disclosure.
3. Confirm every required generated file exists and opens.
4. Restart the kernel and run all cells.
5. Confirm that no error traceback remains and all outputs are visible.
6. Submit the notebook and requested generated files. No separate laboratory report is required.